---
## Section 1 — Core Engine: Nucleotide Base Weights

Each of the four DNA bases (A, C, T, G) acts as a **learnable weight** — the fundamental unit of computation in DigitalDNA, analogous to a neuron's weight in a neural network.

A sequence is processed by summing the weight contributions of each base. The weights start as random values and are optimised during training to best predict the target output.

> **Biological analogy:** Just as binary signals (1s and 0s) propagate through a neural network, nucleotide signals propagate through DigitalDNA — but grounded in the four-letter alphabet of life itself.

In [4]:
import random
import numpy as np
from scipy.optimize import minimize
from itertools import product
from collections import Counter

# ---------------------------------------------------------------
# Nucleotide base weight lookup
# params: dict or array of 4 values [A, C, T, G]
# Each base carries a learned numerical weight — the core
# computational unit of DigitalDNA, analogous to a neuron weight.
# ---------------------------------------------------------------
def process_base(base, params):
    """Return the learned weight for a given nucleotide base.

    Parameters
    ----------
    base : str
        One of 'A', 'C', 'T', 'G'.
    params : dict or array-like
        Learned weights. Dict keys 'A','C','T','G' or array [wA, wC, wT, wG].

    Returns
    -------
    float
        The weight associated with this base.
    """
    if isinstance(params, dict):
        return params.get(base.upper(), 0.0)
    # Array form: index 0=A, 1=C, 2=T, 3=G
    idx = {'A': 0, 'C': 1, 'T': 2, 'G': 3}
    return params[idx[base.upper()]] if base.upper() in idx else 0.0


def process_sequence(sequence, params):
    """Sum the weighted contributions of all bases in a sequence.

    This is the forward pass of DigitalDNA — equivalent to the
    dot product of inputs and weights in a standard linear neuron.

    Parameters
    ----------
    sequence : str
        DNA-encoded feature string, e.g. 'GCATC'.
    params : dict or array-like
        Learned base weights.

    Returns
    -------
    float
        Raw weighted score before epigenetic modulation.
    """
    return sum(process_base(b, params) for b in sequence.upper())


def generate_random_dna_sequence(length):
    """Generate a random DNA sequence of the given length."""
    return ''.join(random.choice(['A', 'C', 'T', 'G']) for _ in range(length))




---
## Section 2 — Epigenetic Modulation Layer

This is the core innovation of DigitalDNA. After computing the raw sequence score, we apply **epigenetic modifiers** — external environmental factors that up- or down-regulate the final output.

In biology, epigenetics explains why identical DNA sequences produce different phenotypes in different environments. In DigitalDNA, the same nucleotide sequence will produce different predictions depending on the context it is evaluated in.

In [5]:
def apply_epigenetics(raw_value, epigenetic_factors, epigenetic_weights):
    """
    Generalized, learnable epigenetic modulation.

    Parameters
    ----------
    raw_value : float
    epigenetic_factors : dict
        Any input features (domain-independent)
    epigenetic_weights : dict
        Learnable weights for each factor

    Returns
    -------
    float
    """
    modifier = 0.0
    for key, value in epigenetic_factors.items():
        modifier += epigenetic_weights.get(key, 0.0) * value

    return raw_value + modifier


---
## Section 3 — Optimisation: Learning the Base Weights

DigitalDNA learns optimal nucleotide weights using **L-BFGS-B**, a quasi-Newton gradient descent method well-suited to small parameter spaces. The cost function minimises squared prediction error across all training sequences.

This is equivalent to training the weight layer of a single-layer neural network — but with the constraint that the weight space is defined by the four nucleotides, keeping the model interpretable and biologically grounded.

In [6]:
def model_predict_binary(sequence, dna_weights, epigenetic_factors, epigenetic_weights, threshold=1.5):
    adjusted = apply_epigenetics(
        process_sequence(sequence, dna_weights),
        epigenetic_factors,
        epigenetic_weights
    )
    return 1 if adjusted > threshold else 0


def model_predict_probability(sequence, dna_weights, epigenetic_factors, epigenetic_weights):
    adjusted = apply_epigenetics(
        process_sequence(sequence, dna_weights),
        epigenetic_factors,
        epigenetic_weights
    )
    return 1.0 / (1.0 + np.exp(-adjusted))


def model_predict_probabilities(sequence, dna_weights, epigenetic_factors, epigenetic_weights):
    adjusted = apply_epigenetics(
        process_sequence(sequence, dna_weights),
        epigenetic_factors,
        epigenetic_weights
    )

    scores = np.array([
        max(0, 4.0 - adjusted),
        max(0, 3.0 - abs(adjusted - 1.5)),
        max(0, 3.0 - abs(adjusted - 2.5)),
        max(0, adjusted - 2.5)
    ])

    total = scores.sum()
    probs = scores / total if total > 0 else np.ones(4) / 4

    return dict(zip(
        ['Uninhabitable', 'Marginal', 'Potentially Habitable', 'Habitable'],
        probs
    ))


def cost_function(all_weights, sequences, expected_outputs, epigenetic_factors_list, epi_keys):
    # Split weights
    dna_weights = {
        'A': all_weights[0],
        'C': all_weights[1],
        'T': all_weights[2],
        'G': all_weights[3]
    }

    epigenetic_weights = {
        key: all_weights[4 + i]
        for i, key in enumerate(epi_keys)
    }

    total_error = 0.0

    for seq, expected, epi in zip(sequences, expected_outputs, epigenetic_factors_list):
        prob = model_predict_probability(seq, dna_weights, epi, epigenetic_weights)
        total_error += (prob - expected) ** 2

    return total_error

def train(sequences, expected_outputs, epigenetic_factors_list, seed=42):
    np.random.seed(seed)

    # Automatically detect ALL epigenetic features
    epi_keys = sorted({
        key for epi in epigenetic_factors_list for key in epi.keys()
    })

    total_params = 4 + len(epi_keys)

    initial_weights = np.random.uniform(-1, 1, total_params)

    result = minimize(
        fun=cost_function,
        x0=initial_weights,
        args=(sequences, expected_outputs, epigenetic_factors_list, epi_keys),
        method='L-BFGS-B',
        options={'maxiter': 1000}
    )

    dna_weights = {
        'A': result.x[0],
        'C': result.x[1],
        'T': result.x[2],
        'G': result.x[3]
    }

    epigenetic_weights = {
        key: result.x[4 + i]
        for i, key in enumerate(epi_keys)
    }

    print(f"Training complete. Final cost: {result.fun:.4f}")
    print(f"DNA weights: {dna_weights}")
    print(f"Epigenetic weights: {epigenetic_weights}")

    return dna_weights, epigenetic_weights

---
## Section 4 — Evolutionary Simulation

DigitalDNA incorporates a **genetic algorithm** that evolves a population of DNA sequences over generations. Sequences with higher fitness scores are selected to reproduce, with mutation introducing variation — directly modelling Darwinian natural selection.

In an astrobiology context, this can be used to **evolve optimal biosignature profiles** — discovering which nucleotide-encoded atmospheric compositions maximise habitability scores under a given set of planetary epigenetic conditions.

In [7]:
def run_evolution(pop_size, seq_length, generations,
                  epigenetic_factors,
                  dna_weights,
                  epigenetic_weights,
                  mutation_rate=0.05,
                  verbose=True):

    population = [
        ''.join(random.choice(['A','C','T','G']) for _ in range(seq_length))
        for _ in range(pop_size)
    ]

    for gen in range(generations):

        fitness_scores = [
            model_predict_probability(seq, dna_weights, epigenetic_factors, epigenetic_weights)
            for seq in population
        ]

        # Select top 50%
        top_indices = np.argsort(fitness_scores)[::-1][:pop_size // 2]
        top_sequences = [population[i] for i in top_indices]

        # Reproduce
        new_population = []
        while len(new_population) < pop_size:
            parent = random.choice(top_sequences)
            new_population.append(reproduce(parent, mutation_rate))

        population = new_population

        if verbose:
            print(f"Generation {gen+1:>3} | Best fitness: {max(fitness_scores):.4f}")

    return population, fitness_scores